
# Trích xuất feature từ patch mô học

Notebook này chạy bộ trích xuất đã đóng gói trong `histology-feature-runtime.zip`. Nó đọc một nguồn đã chọn (thư mục PNG hoặc ZIP), chạy lần lượt trên dữ liệu chưa có commit, rồi ghi feature, chỉ mục, QC và trạng thái vào Drive. Có thể chạy lại cùng cấu hình và cùng thư mục output sau khi tải thêm ZIP hoặc Colab bị ngắt.

Bắt đầu với `RUN_MODE = 'smoke'`; xem lại output smoke trước khi chuyển sang `production`. Chọn T4 trong giao diện Colab nếu muốn GPU. Notebook chỉ dùng GPU phiên hiện tại, không tạo hay tắt VM.


In [ ]:

# Chỉ thư viện chuẩn trước khi runtime bundle được giải nén.
import importlib
import importlib.util
import json
import shutil
import sys
import zipfile
from pathlib import Path, PurePosixPath

IS_COLAB = (
    importlib.util.find_spec('google') is not None
    and importlib.util.find_spec('google.colab') is not None
)
if IS_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/histology')
else:
    # Khi chạy Jupyter cục bộ, sửa ROOT thành thư mục histology của máy này.
    ROOT = Path.home() / 'histology'

RUNTIME_ZIP = ROOT / 'runtime' / 'histology-feature-runtime.zip'
RUNTIME_ROOT = Path('/content/histology-feature-runtime') if IS_COLAB else Path('/tmp/histology-feature-runtime')
print('Environment:', 'Colab' if IS_COLAB else 'local Jupyter')
print('Drive/data root:', ROOT)



## Chọn nguồn và chế độ chạy

Chọn đúng một `SOURCE_KIND` cho mỗi lượt. ZIP mode đọc các archive đã tải vào `source/archives`; không cần chờ đủ 25 ZIP mới bắt đầu. Directory mode đọc PNG từ một thư mục đã có sẵn. Không giải nén rồi nén lại các ZIP chỉ để đổi định dạng.

`smoke` ghi vào namespace riêng, mỗi lượt tối đa một part mới và sáu patch để kiểm tra đường chạy. Danh sách part smoke được lưu; rerun cùng nguồn giữ đúng mẫu đó. Sau khi xem QC và commit smoke, đổi sang `production`; production có thư mục output riêng và không đặt giới hạn patch. Rerun production dùng đúng output cũ để xác minh commit rồi tiếp tục.

Các biến `BATCH_SIZE`, `DEVICE`, `PRECISION`, `BUDGET_MINUTES`, `RESERVE_MINUTES`, `RECOVER_STALE_LOCK` và `PRODUCTION_MAX_NEW_PARTS` nằm trong cell cấu hình bên dưới để người vận hành chỉnh trực tiếp.

In [ ]:

SOURCE_KIND = 'zip'  # Chỉ chọn 'zip' hoặc 'directory'.
ZIP_ROOT = ROOT / 'source' / 'archives'
PNG_DIRECTORY = ROOT / 'source' / 'tiles'  # Sửa nếu thư mục PNG nằm nơi khác.
DIRECTORY_ALIASES = None  # Chỉ cấu hình alias đã xác minh khi dùng directory; xem hướng dẫn.
SOURCE_ROOT = ZIP_ROOT if SOURCE_KIND == 'zip' else PNG_DIRECTORY
METADATA = ROOT / 'source' / 'Metadata.xlsx'

RUN_MODE = 'smoke'  # Đổi thành 'production' sau khi đã review smoke.
BATCH_SIZE = 32
DEVICE = 'cuda'  # Đổi thành 'cpu' nếu chạy local không có GPU.
PRECISION = 'fp32'  # Có thể dùng 'amp_fp16' trên CUDA nếu đã chủ động kiểm tra.
BUDGET_MINUTES = 240
RESERVE_MINUTES = 30
RECOVER_STALE_LOCK = False  # Chỉ bật sau khi xác nhận runtime cũ đã dừng.
PRODUCTION_MAX_NEW_PARTS = None  # Đặt số dương nếu muốn giới hạn một lượt production.
FEATURES_ROOT = ROOT / 'features'
WORK_ROOT = Path('/content/histology-feature-work') if IS_COLAB else Path('/tmp/histology-feature-work')
WEIGHTS_DIR = ROOT / 'encoders' / 'resnet50_imagenet_v1'

if SOURCE_KIND not in {'zip', 'directory'}:
    raise ValueError("SOURCE_KIND phải là 'zip' hoặc 'directory'.")
if RUN_MODE not in {'smoke', 'production'}:
    raise ValueError("RUN_MODE phải là 'smoke' hoặc 'production'.")
if not METADATA.is_file():
    raise FileNotFoundError(f'Không thấy Metadata.xlsx: {METADATA}')
if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError(f'Không thấy nguồn {SOURCE_KIND}: {SOURCE_ROOT}')
if SOURCE_KIND == 'zip':
    available_sources = sorted(SOURCE_ROOT.glob('Tiles-*.zip'))
    print(f'ZIP hiện có: {len(available_sources)}. Thiếu ZIP không chặn lượt chạy từng phần.')
else:
    print('Nguồn PNG:', SOURCE_ROOT)

print('Output mode:', RUN_MODE)
print('Nguồn được chọn:', SOURCE_ROOT)


In [ ]:

if not RUNTIME_ZIP.is_file():
    raise FileNotFoundError(
        f'Không thấy runtime mới: {RUNTIME_ZIP}. Hãy tải histology-feature-runtime.zip vào '
        'histology/runtime; histology-data-runtime.zip cũ không chứa feature extractor.'
    )

staging_root = RUNTIME_ROOT.with_name(RUNTIME_ROOT.name + '.staging')
shutil.rmtree(staging_root, ignore_errors=True)
staging_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(RUNTIME_ZIP) as bundle:
    for member in bundle.infolist():
        relative = PurePosixPath(member.filename)
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError(f'Đường dẫn không an toàn trong runtime ZIP: {member.filename}')
        destination = staging_root.joinpath(*relative.parts)
        if member.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        with bundle.open(member) as source, destination.open('wb') as target:
            shutil.copyfileobj(source, target)

if not (staging_root / 'histology_data' / '__init__.py').is_file():
    raise RuntimeError('Runtime ZIP thiếu package histology_data.')
if RUNTIME_ROOT.exists():
    shutil.rmtree(RUNTIME_ROOT)
staging_root.replace(RUNTIME_ROOT)
for module_name in list(sys.modules):
    if module_name == 'histology_data' or module_name.startswith('histology_data.'):
        del sys.modules[module_name]
if str(RUNTIME_ROOT) not in sys.path:
    sys.path.insert(0, str(RUNTIME_ROOT))
CONFIG_PATH = RUNTIME_ROOT / 'configs' / 'features_colab.json'
if not CONFIG_PATH.is_file():
    raise RuntimeError(f'Runtime ZIP thiếu cấu hình: {CONFIG_PATH}')
with CONFIG_PATH.open(encoding='utf-8') as stream:
    CONFIG = json.load(stream)
print('Đã giải nén runtime:', RUNTIME_ROOT)



## Kiểm tra môi trường rồi chạy

Notebook không cài lại PyTorch/torchvision và không tải code từ mạng. Colab thường đã có cặp thư viện tương thích. Nếu chạy cục bộ, cài requirements của runtime trong virtual environment trước; giữ cặp PyTorch/torchvision phù hợp với CPU hoặc CUDA của máy.

Nếu cache ResNet50 V1 chưa có, lượt đầu có Internet tự tải checkpoint chính thức (khoảng 100 MB), xác minh SHA-256 `0676ba61b6795bbe1773cffd859882e5e297624d384b6993f7c9e683e722fb8a`, rồi cache vào `WEIGHTS_DIR/resnet50_imagenet1k_v1/`. Không cần chuẩn bị sẵn weights khi có Internet. Khi chạy offline, chuẩn bị trước checkpoint và `manifest.json` đã xác minh trong thư mục cache này.

Mặc định ngân sách là 240 phút, dành 30 phút cuối cho commit và báo cáo. Đặt ngân sách theo thời gian còn được phép của phiên, tính cả thời gian khởi động và thao tác Drive. Khi chạy Colab, người vận hành tự chọn GPU T4 trong giao diện.

Khóa output bảo vệ lượt ghi đang chạy. Chỉ bật `RECOVER_STALE_LOCK = True` sau khi đã xác nhận runtime trước dừng hẳn.


In [ ]:

try:
    import torch
    import torchvision
except Exception as exc:
    raise RuntimeError(
        'Không import được PyTorch/torchvision hiện có. Hãy cài cặp phiên bản tương thích trong môi trường '
        'local hoặc chọn Colab GPU phù hợp; notebook không tự thay đổi các package GPU.'
    ) from exc

print('PyTorch / torchvision:', torch.__version__, '/', torchvision.__version__)
if DEVICE.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError(
        'Cấu hình yêu cầu CUDA nhưng phiên hiện tại không có GPU. Hãy chọn GPU trong giao diện Colab, '
        'hoặc đặt DEVICE=cpu nếu chủ động chạy local trên CPU.'
    )
if DEVICE.startswith('cuda'):
    print('GPU hiện tại:', torch.cuda.get_device_name(0))
else:
    print('Thiết bị:', DEVICE)

features = importlib.import_module('histology_data.features')
FeatureRunConfig = features.FeatureRunConfig
run_feature_extraction = features.run_feature_extraction
if RUN_MODE == 'smoke':
    OUTPUT_ROOT = FEATURES_ROOT / 'smoke_v001'
    MAX_NEW_PARTS = 1
    MAX_PATCHES_PER_PART = 6
else:
    OUTPUT_ROOT = FEATURES_ROOT / 'v001'
    MAX_NEW_PARTS = PRODUCTION_MAX_NEW_PARTS
    MAX_PATCHES_PER_PART = None

run_values = dict(CONFIG)
run_values.update(
    metadata=str(METADATA),
    source_root=str(SOURCE_ROOT),
    source_kind=SOURCE_KIND,
    aliases=DIRECTORY_ALIASES if SOURCE_KIND == 'directory' else None,
    output_root=str(OUTPUT_ROOT),
    work_root=str(WORK_ROOT),
    weights_dir=str(WEIGHTS_DIR),
    batch_size=BATCH_SIZE,
    device=DEVICE,
    precision=PRECISION,
    max_new_parts=MAX_NEW_PARTS,
    max_patches_per_part=MAX_PATCHES_PER_PART,
    budget_minutes=BUDGET_MINUTES,
    reserve_minutes=RESERVE_MINUTES,
    recover_lock=RECOVER_STALE_LOCK,
)
run_config = FeatureRunConfig.from_dict(run_values)
print('Output:', OUTPUT_ROOT)
print('Giới hạn part mới / patch mỗi part:', MAX_NEW_PARTS, '/', MAX_PATCHES_PER_PART)


In [ ]:

FEATURE_RESULT = run_feature_extraction(run_config)
print(json.dumps(FEATURE_RESULT, ensure_ascii=False, indent=2, default=str))
print('Xem status.json, QC và commit.json trong output trước khi quyết định bước tiếp theo.')



## Sau lượt chạy

Xem `status.json`, báo cáo QC, index, feature arrays và commit trong thư mục output. Chỉ các part đã commit mới được tính là hoàn tất. `complete` nghĩa là đủ nguồn theo cấu hình; `smoke_complete` là smoke đã chạy xong; `part_limit_reached` là giới hạn part của lượt đã chạm (có thể là smoke có chủ đích); `awaiting_sources` cho biết có thể tiếp tục khi upload thêm; `budget_exhausted` cho biết lượt chạy dừng theo giới hạn thời gian. Lỗi checksum hoặc dữ liệu phải được điều tra trước khi dùng lại output.

Feature vectors là biểu diễn ảnh của encoder đã cố định; chúng không phải nhãn bệnh hay kết luận ung thư. Nhãn trong Metadata.xlsx chỉ ở cấp ca và cần review; patient identity chưa được xác minh. Mọi output hiện vẫn `training_ready=false`; notebook này chưa tạo bag, MIL hoặc mô hình dự đoán. Khi đã `complete`, đang chờ nguồn, hoặc gặp lỗi, người vận hành tự ngắt/xóa runtime Colab để dừng GPU idle.
